In [1]:
"""
Dixon Matrix Size computing using SageMath
"""

import time
from sage.all import *

def get_boundary_heights(d_list, shift=-1, include_prefix_zero=True):
    """
    Unified boundary height calculation function
    
    Args:
        d_list: list of degrees
        shift: slope correction (Dixon typically uses -1)
        include_prefix_zero: whether to add a[0] = 0 at the beginning
            - True:  a = [0, s[0], s[0]+s[1], ...] (for determinant/Hessenberg)
            - False: a = [s[0], s[0]+s[1], ...]     (for DP)
    
    Returns:
        (a, slopes): boundary array and sorted slopes
    """
    n = len(d_list)
    
    # 1. Calculate slopes and sort in descending order
    slopes = [d + shift for d in d_list]
    slopes.sort(reverse=True)
    
    # 2. Calculate upper bound a
    if include_prefix_zero:
        # Determinant/Hessenberg mode: a = [0, s[0], s[0]+s[1], ...]
        a = [0]
        current_h = 0
        for i in range(n - 1):
            current_h += slopes[i]
            a.append(current_h)
    else:
        # DP mode: a = [s[0], s[0]+s[1], ...]
        a = []
        current_h = 0
        for i in range(n):
            current_h += slopes[i]
            a.append(current_h)
    
    return a, slopes


def dixon_size_determinant(d_list):
    """
    Method 1: Determinant formula (Theorem 10.7.1)
    
    Computes the exact count using the determinant of a binomial coefficient matrix.
    Time complexity: O(n^3) for matrix determinant
    
    Args:
        d_list: list of polynomial degrees
        shift: slope correction parameter
    
    Returns:
        Integer: exact count of lattice paths
    """
    n = len(d_list)
    shift = -1
    a, slopes = get_boundary_heights(d_list, shift, include_prefix_zero=True)
    
    # Create matrix
    M = Matrix(ZZ, n, n)

    for i in range(n):
        for j in range(n):
            # entry(i, j) = binomial(a[i] + 1, j - i + 1)
            upper = a[i] + 1
            lower = j - i + 1
            
            if lower < 0 or lower > upper:
                M[i, j] = 0
            else:
                M[i, j] = binomial(upper, lower)

    return M.determinant()


def dixon_size_Hessenberg(d_list):
    """
    Method 2: Hessenberg recurrence - O(n^2)
    
    Uses the upper Hessenberg structure of the matrix to compute the determinant
    more efficiently via recurrence relation.
    
    Args:
        d_list: list of polynomial degrees
        shift: slope correction parameter
    
    Returns:
        Integer: exact count of lattice paths
    """
    n = len(d_list)
    shift = -1
    a, slopes = get_boundary_heights(d_list, shift, include_prefix_zero=True)
    
    # D[k] stores the determinant of the first k x k submatrix
    D = [Integer(1)]  # D[0] = 1
    
    for k in range(1, n + 1):
        sum_val = Integer(0)
        
        for i in range(1, k + 1):
            row_idx = i - 1
            col_idx = k - 1
            
            # Binomial coefficient: binomial(a[row_idx] + 1, col_idx - row_idx + 1)
            upper = a[row_idx] + 1
            lower = col_idx - row_idx + 1
            
            if lower >= 0 and lower <= upper:
                m_val = binomial(upper, lower)
            else:
                m_val = Integer(0)
            
            # Sign: (-1)^(k-i)
            term = m_val * D[i - 1]
            if (k - i) % 2 == 1:
                term = -term
            
            sum_val += term
        
        D.append(sum_val)
    
    return D[n]

def dixon_size_dp(d_list):
    """
    Method 3: Dynamic programming
    
    Computes lattice paths using DP with prefix sum optimization.
    Time complexity: O(n * H) where H is the maximum height
    
    Args:
        d_list: list of polynomial degrees
        shift: slope correction parameter
    
    Returns:
        Integer: exact count of lattice paths
    """
    n = len(d_list)
    shift = -1
    a, slopes = get_boundary_heights(d_list, shift, include_prefix_zero=False)
    
    max_height = a[n - 1]
    
    # Initialize DP array
    current_dp = [Integer(0)] * (max_height + 1)
    
    # Step 1 initialization
    for h in range(a[0] + 1):
        current_dp[h] = Integer(1)
    
    # Subsequent steps
    for k in range(1, n):
        upper_bound = a[k]
        new_dp = [Integer(0)] * (max_height + 1)
        
        # Use prefix sum optimization
        current_sum = Integer(0)
        for h in range(min(upper_bound, max_height) + 1):
            current_sum += current_dp[h]
            new_dp[h] = current_sum
        
        current_dp = new_dp
    
    return current_dp[a[n - 1]]

def Unrestricted_bound(d_list):
    """
    Upper bound: Unrestricted bound
    
    Without considering concave boundary constraints, only considers all monotone paths
    from (0,0) to (n, D) where D = sum(d_i + shift).
    Number of paths = binomial(D+n, n)
    
    Args:
        d_list: list of polynomial degrees
        shift: slope correction parameter
    
    Returns:
        Integer: upper bound on lattice path count
    """
    sorted_d = sorted(d_list)  
    new_d_list = sorted_d[1:] 
    shift = -1
    n = len(new_d_list)
    total_height = sum(d + shift for d in new_d_list)
    
    if total_height < 0:
        return Integer(0)
    
    return binomial(total_height + n, n)

def dixon_size_fuss_catalan(n, d):
    """Compute the Fuss-Catalan number D = (1/(n(d-1)+1)) * C(nd, n)"""
    if n == 0 or d == 0:
        return 1
    return binomial(d*n, n) / ((d-1)*n + 1)

def dixon_complexity(d_list, n, omega):
    """
    Compute Dixon resultant complexity estimate.
    
    Parameters:
    - d_values: Degree sequence
    - n: Number of variables
    - omega: Matrix multiplication exponent
    
    Returns: log₂ of complexity
    """
    size = dixon_size_Hessenberg(d_list)
    d = 0
    m = len(d_list)
    
    if m == n + 1:
        d = 1
    elif m == n:
        d = sum(d_list)
    elif m < n:
        d = (sum(d_list) + 1) ** (n - m + 1)
    else:
        return 0
    
    return log(d * (size) ** omega, 2).n(digits = 5)

def macaulay_bound(n, d):
    """Macaulay bound: d_reg <= 1 + n(d-1)"""
    return 1 + n*(d-1)

def gb_classical_complexity(n, d, omega):
    """
    (Classical) Gröbner basis complexity (classical bound mentioned by reviewer #727A)
    O(binom(n + d_reg, d_reg)^omega)
    """
    d_reg = macaulay_bound(n, d)
    return log(binomial(n + d_reg, d_reg)^omega,2).n()

def gb_bariant_formula(n, d, omega):
    """
    Formula by (F5 refined) mentioned by reviewer #727C:
    O(n * d_reg * binom(n + d_reg - 1, d_reg)^omega)
    """
    d_reg = macaulay_bound(n, d)
    return log(n * d_reg * binomial(n + d_reg - 1, d_reg)^omega,2).n()


def test_dixon_size():
    """Test Dixon matrix size estimation methods."""
    
    print("Test 1: Varying degrees [5,6,7,9,11]")
    omega = 2
    d_list = [5, 6, 7, 9, 11]
    
    print(f"  Determinant: {dixon_size_determinant(d_list)}")
    print(f"  Hessenberg: {dixon_size_Hessenberg(d_list)}")
    print(f"  DP: {dixon_size_dp(d_list)}")
    print(f"  Bound: {Unrestricted_bound(d_list)}")
    print(f"  Complexity: {dixon_complexity(d_list, len(d_list), omega)}")
    
    print("\nTest 2: Homogeneous 20-degree, 10 variables")
    n = 10
    d = 20
    d_list = [d] * n
    
    print(f"  Fuss-Catalan: {dixon_size_fuss_catalan(n, d)}")
    print(f"  Determinant: {dixon_size_determinant(d_list)}")
    print(f"  Hessenberg: {dixon_size_Hessenberg(d_list)}")
    print(f"  DP: {dixon_size_dp(d_list)}")
    print(f"  Bound: {Unrestricted_bound(d_list)}")
    print(f"  Complexity: {dixon_complexity(d_list, n, omega)}")

    print("\nTest 3: Homogeneous 2-degree, 10 variables")
    n = 5
    d = 10
    d_list = [d] * n
    
    print(f"  Fuss-Catalan: {dixon_size_fuss_catalan(n, d)}")
    print(f"  Determinant: {dixon_size_determinant(d_list)}")
    print(f"  Hessenberg: {dixon_size_Hessenberg(d_list)}")
    print(f"  DP: {dixon_size_dp(d_list)}")
    print(f"  Bound: {Unrestricted_bound(d_list)}")
    print(f"  Complexity: {dixon_complexity(d_list, n, omega)}")    
    print(f"  Gronber Complexity: {gb_bariant_formula(n, d, omega)}") 
    
    
    print("\nTest 4: Homogeneous 2-degree, 20 variables")
    n = 3
    d = 20
    d_list = [d] * n
    
    print(f"  Fuss-Catalan: {dixon_size_fuss_catalan(n, d)}")
    print(f"  Determinant: {dixon_size_determinant(d_list)}")
    print(f"  Hessenberg: {dixon_size_Hessenberg(d_list)}")
    print(f"  DP: {dixon_size_dp(d_list)}")
    print(f"  Bound: {Unrestricted_bound(d_list)}")
    print(f"  Complexity: {dixon_complexity(d_list, n, omega)}")    
    print(f"  Gronber Complexity: {gb_bariant_formula(n, d, omega)}") 
    
    
if __name__ == "__main__":
    test_dixon_size()

Test 1: Varying degrees [5,6,7,9,11]
  Determinant: 28149
  Hessenberg: 28149
  DP: 28149
  Bound: 40920
  Complexity: 34.810

Test 2: Homogeneous 20-degree, 10 variables
  Fuss-Catalan: 117544525178080
  Determinant: 117544525178080
  Hessenberg: 117544525178080
  DP: 117544525178080
  Bound: 446098010817800
  Complexity: 101.12

Test 3: Homogeneous 2-degree, 10 variables
  Fuss-Catalan: 46060
  Determinant: 46060
  Hessenberg: 46060
  DP: 46060
  Bound: 91390
  Complexity: 36.626
  Gronber Complexity: 43.4717998220795

Test 4: Homogeneous 2-degree, 20 variables
  Fuss-Catalan: 590
  Determinant: 590
  Hessenberg: 590
  DP: 590
  Bound: 780
  Complexity: 24.316
  Gronber Complexity: 29.0220107857894


In [2]:
print("Test: Varying degrees [5,6,7,9,11]")
omega = 2
d_list = [2,3,4,5]

print(f"  Determinant: {dixon_size_determinant(d_list)}")
print(f"  Hessenberg: {dixon_size_Hessenberg(d_list)}")
print(f"  DP: {dixon_size_dp(d_list)}")
print(f"  Bound: {Unrestricted_bound(d_list)}")
print(f"  Complexity: {dixon_complexity(d_list, len(d_list), omega)}")

Test: Varying degrees [5,6,7,9,11]
  Determinant: 170
  Hessenberg: 170
  DP: 170
  Bound: 220
  Complexity: 18.626
